<a href="https://colab.research.google.com/github/HiroTED/sample-aws/blob/main/Claude_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 必要なパッケージをインストール
!pip install -q crewai crewai-tools openai langchain-openai chromadb pypdf python-docx
print("✓ インストール完了!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 701.4/701.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.8/766.8 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 723.4/723.4 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 90.1 MB

In [ ]:
import os
from google.colab import userdata

In [ ]:
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("✓ APIキーを設定しました")
except:
    # または直接入力(非推奨 - ノートブック共有時に注意)
    os.environ['OPENAI_API_KEY'] = 'your-api-key-here'
    print("⚠️ APIキーを直接設定しました。セキュリティに注意してください")


✓ APIキーを設定しました


In [ ]:
# documentsフォルダを作成
!mkdir -p documents

# サンプルドキュメントを作成
with open('documents/python_basics.txt', 'w', encoding='utf-8') as f:
    f.write("""
Pythonの基礎知識

1. 変数の宣言
Pythonでは型を明示せずに変数を宣言できます。
例: name = "太郎"、age = 25

2. リスト
複数の値を格納できるデータ構造です。
例: fruits = ["りんご", "バナナ", "オレンジ"]

3. 関数
def キーワードで関数を定義します。
例: def greet(name):
        return f"こんにちは、{name}さん!"

4. ループ
for文とwhile文があります。
例: for i in range(5):
        print(i)
""")

with open('documents/crewai_memo.txt', 'w', encoding='utf-8') as f:
    f.write("""
CrewAIについてのメモ

CrewAIは複数のAIエージェントを協調させて
複雑なタスクを実行できるフレームワークです。

主要な概念:
- Agent: 特定の役割を持つAI
- Task: エージェントが実行する具体的な作業
- Crew: エージェントのチームを管理
- Tools: エージェントが使用できる機能

利点:
- 複雑なタスクを分割して処理
- エージェント間の協力
- 柔軟な設定
""")

print("✓ サンプルドキュメントを作成しました")
print("📁 作成されたファイル:")
!ls -la documents/

✓ サンプルドキュメントを作成しました
📁 作成されたファイル:
total 16
drwxr-xr-x 2 root root 4096 Jan  9 05:18 .
drwxr-xr-x 1 root root 4096 Jan  9 05:18 ..
-rw-r--r-- 1 root root  470 Jan  9 05:18 crewai_memo.txt
-rw-r--r-- 1 root root  489 Jan  9 05:18 python_basics.txt


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import DirectoryReadTool, FileReadTool

# ツールの設定
docs_tool = DirectoryReadTool(directory='./documents')
file_tool = FileReadTool()

# Agent 1: ドキュメント検索エージェント
search_agent = Agent(
    role='ナレッジ検索スペシャリスト',
    goal='ユーザーの質問に関連するドキュメントを見つける',
    backstory="""あなたは優秀な情報検索の専門家です。
    ユーザーの質問を理解し、最も関連性の高いドキュメントを
    素早く見つけ出すことができます。""",
    tools=[docs_tool, file_tool],
    verbose=True,
    allow_delegation=False
)

# Agent 2: 回答生成エージェント
answer_agent = Agent(
    role='回答作成スペシャリスト',
    goal='検索結果を元に、わかりやすく正確な回答を作成する',
    backstory="""あなたは親切で知識豊富なアシスタントです。
    見つかった情報を整理し、ユーザーにとって理解しやすい
    形で回答を提供します。""",
    verbose=True,
    allow_delegation=False
)

print("✓ エージェントを作成しました!")

✓ エージェントを作成しました!


In [ ]:
def ask_question(question):
    """質問を受け取り、回答を返す関数"""

    # タスク1: ドキュメント検索
    search_task = Task(
        description=f"""
        以下の質問に関連するドキュメントを検索してください:
        質問: {question}

        documentsフォルダ内のファイルから関連情報を見つけ、
        重要な内容を抽出してください。
        """,
        agent=search_agent,
        expected_output="質問に関連するドキュメントの内容と重要なポイント"
    )

    # タスク2: 回答生成
    answer_task = Task(
        description=f"""
        検索エージェントが見つけた情報を元に、以下の質問に回答してください:
        質問: {question}

        回答は:
        - 簡潔でわかりやすく
        - 根拠となる情報源を明記
        - 必要に応じて補足説明を追加
        """,
        agent=answer_agent,
        expected_output="ユーザーの質問に対する明確で有用な回答",
        context=[search_task]
    )

    # Crewを作成して実行
    crew = Crew(
        agents=[search_agent, answer_agent],
        tasks=[search_task, answer_task],
        process=Process.sequential,
        verbose=True
    )

    result = crew.kickoff()
    return result

print("✓ 質問応答システムの準備完了!")

✓ 質問応答システムの準備完了!


In [ ]:
# 質問を入力
question = "Pythonの変数について教えて"

print(f"🤔 質問: {question}\n")
print("🔍 AI Agentが処理中...\n")
print("="*50)

# 回答を取得
answer = ask_question(question)

print("="*50)
print("\n✅ 回答:")
print(answer)

🤔 質問: Pythonの変数について教えて

🔍 AI Agentが処理中...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  4ee9af77-86cd-4b8f-97b1-cdc51f8ea73c                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          以下の質問に関連するドキュメントを検索してください:                                                    │
│          質問: Pythonの変数について教えて                                                                       │
│                                                                                                                 │
│          documentsフォルダ内のファイルから関連情報を見つけ、                                                    │
│          重要な内容を抽出してください。                                                                         │
│                                                                                                                 │
│  ID: cccafb87-2a76-4b18-8102-7cc6634b19bc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: ナレッジ検索スペシャリスト                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│          以下の質問に関連するドキュメントを検索してください:                                                    │
│          質問: Pythonの変数について教えて                                                                       │
│                                                                                                                 │
│          documentsフォルダ内のファイルから関連情報を見つけ、                                                    │
│          重要な内容を抽出してください。                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


...

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│                                                                                                                 │
│          以下の質問に関連するドキュメントを検索してください:                                                    │
│          質問: Pythonの変数について教えて                                                                       │
│                                                                                                                 │
│          documentsフォルダ内のファイルから関連情報を見つけ、                                                    │
│          重要な内容を抽出してください。                                                                         │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  ナレッジ検索スペシャリスト                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# 質問を入力
question = "Pythonの変数について教えて"

print(f"🤔 質問: {question}\n")
print("🔍 AI Agentが処理中...\n")
print("="*50)

# 回答を取得
answer = ask_question(question)

print("="*50)
print("\n✅ 回答:")
print(answer)

🤔 質問: Pythonの変数について教えて

🔍 AI Agentが処理中...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  8a925664-edd0-4d50-a9e1-3daf002a38d7                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          以下の質問に関連するドキュメントを検索してください:                                                    │
│          質問: Pythonの変数について教えて                                                                       │
│                                                                                                                 │
│          documentsフォルダ内のファイルから関連情報を見つけ、                                                    │
│          重要な内容を抽出してください。                                                                         │
│                                                                                                                 │
│  ID: b862802b-dba1-4c7c-bb04-83c02a77cdab                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: ナレッジ検索スペシャリスト                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│          以下の質問に関連するドキュメントを検索してください:                                                    │
│          質問: Pythonの変数について教えて                                                                       │
│                                                                                                                 │
│          documentsフォルダ内のファイルから関連情報を見つけ、                                                    │
│          重要な内容を抽出してください。                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: List files in directory                                                                                  │
│  Args: {}                                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 回答作成スペシャリスト                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Pythonの変数について簡潔に説明します。                                                                         │
│                                                                                                                 │
│  1. **変数の宣言方法**                                                                                          │
│  Pythonでは変数を宣言する際に型を明示する必要はありません。値を代入するだけで変数が作成されます。               │
│  例:                                                                                                            │
│  ```python                                                                                                      │
│  name = "太郎"                                                                                                  │
│  age = 25                                                                                                       │
│  ```                                                                                                            │
│  ここでは`name`に文字列、`age`に整数を代入しています。                                                          │
│                                                                                                                 │
│  2. **データ型の自動判別**                                                                                      │
│  Pythonは代入された値の型を自動で判断するため、例えば同じ変数に異なる型の値を代入し直すことも可能です。         │
│  例:                                                                                                            │
│  ```python                                                                                                      │
│  x = 10    # 整数型                                                                                             │
│  x = "猫"  # 文字列型に変更                                                                                     │
│  ```                                                                                                            │
│                                                                                                                 │
│  3. **補足: 変数名のルール**                                                                                    │
│  - 英数字とアンダースコア `_` を使う                                                                            │
│  - 数字から始めてはいけない                                                                                     │
│  - 大文字と小文字は区別される（`age` と `Age` は別の変数）                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  【根拠】                                                                                                       │
│  提供された「Pythonの基礎知識」情報より「1. 変数の宣言」部分を参考。                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  補足として、変数はプログラム内でデータを一時的に保存し操作するための名前付き入れ物として使われる基本的な構成   │
│  要素です。Pythonの特徴として簡潔な記

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│          検索エージェントが見つけた情報を元に、以下の質問に回答してください:                                    │
│          質問: Pythonの変数について教えて                                                                       │
│                                                                                                                 │
│          回答は:                                                                                                │
│          - 簡潔でわかりやすく                                                                                   │
│          - 根拠となる情報源を明記                                                                               │
│          - 必要に応じて補足説明を追加                                                                           │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  回答作成スペシャリスト                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ 回答:
Pythonの変数について簡潔に説明します。

1. **変数の宣言方法**  
Pythonでは変数を宣言する際に型を明示する必要はありません。値を代入するだけで変数が作成されます。  
例:  
```python
name = "太郎"
age = 25
```
ここでは`name`に文字列、`age`に整数を代入しています。

2. **データ型の自動判別**  
Pythonは代入された値の型を自動で判断するため、例えば同じ変数に異なる型の値を代入し直すことも可能です。  
例:  
```python
x = 10    # 整数型
x = "猫"  # 文字列型に変更
```

3. **補足: 変数名のルール**  
- 英数字とアンダースコア `_` を使う  
- 数字から始めてはいけない  
- 大文字と小文字は区別される（`age` と `Age` は別の変数）

---

【根拠】  
提供された「Pythonの基礎知識」情報より「1. 変数の宣言」部分を参考。

---

補足として、変数はプログラム内でデータを一時的に保存し操作するための名前付き入れ物として使われる基本的な構成要素です。Pythonの特徴として簡潔な記法と動的型付け（実行時に型が決まる）が挙げられます。これにより、初心者でも扱いやすくなっています。


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

In [ ]:
from unstructured.partition.auto import partition
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

class HybridKnowledgeBase:
    def __init__(self, docs_path='./documents'):
        self.docs_path = docs_path
        self.embeddings = OpenAIEmbeddings()
        self.vectorstore = None

    def build_index(self):
        """Unstructuredで読み込み + LangChainで検索準備"""
        documents = []

        # ✅ Unstructuredで強力なファイル読み込み
        for filename in os.listdir(self.docs_path):
            filepath = os.path.join(self.docs_path, filename)

            # あらゆる形式に対応!
            elements = partition(filename=filepath)
            text = "\n".join([str(el) for el in elements])

            # LangChain形式に変換
            doc = Document(
                page_content=text,
                metadata={"source": filename}
            )
            documents.append(doc)

        # ✅ LangChainでベクトル化・保存
        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            persist_directory="./chroma_db"
        )

        print(f"✓ {len(documents)}個のドキュメントをインデックス化")

    def search(self, query, k=3):
        """LangChainで検索"""
        if not self.vectorstore:
            self.build_index()

        results = self.vectorstore.similarity_search(query, k=k)
        return results

ModuleNotFoundError: No module named 'unstructured'

In [ ]:
import os
import chromadb
from unstructured_client import UnstructuredClient
from unstructured_client.models import shared
from sentence_transformers import SentenceTransformer
from crewai import Agent, Task, Crew, Process

class UnstructuredAPIKnowledgeBase:
    """Unstructured APIを使った知識ベース"""

    def __init__(self, docs_path='./documents'):
        self.docs_path = docs_path

        # Unstructured APIクライアント初期化
        self.unstructured_client = UnstructuredClient(
            api_key_auth=os.getenv("Q8FjAE9XG21BcMVyt0KQjdaQlVjkVA")
        )

        # ローカルEmbeddingモデル(無料!)
        print("🤖 Embeddingモデルをロード中...")
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

        # ChromaDB
        self.chroma_client = chromadb.PersistentClient(path="./chroma_db")
        self.collection = None

    def build_index(self):
        """Unstructured APIでファイル処理 → インデックス化"""
        print("📚 Unstructured APIでドキュメントを処理中...\n")

        # コレクション準備
        try:
            self.chroma_client.delete_collection("documents")
        except:
            pass

        self.collection = self.chroma_client.create_collection(
            name="documents",
            metadata={"description": "知識ベース"}
        )

        all_documents = []
        all_embeddings = []
        all_metadatas = []
        all_ids = []

        # 各ファイルを処理
        for filename in os.listdir(self.docs_path):
            filepath = os.path.join(self.docs_path, filename)

            try:
                print(f"  🔄 処理中: {filename}")

                # Unstructured APIでファイルを送信
                with open(filepath, "rb") as f:
                    req = shared.PartitionParameters(
                        files=shared.Files(
                            content=f.read(),
                            file_name=filename,
                        ),
                        strategy="hi_res",  # 高精度モード
                        languages=["jpn", "eng"],  # 日本語・英語対応
                    )

                # APIでパース実行
                res = self.unstructured_client.general.partition(req)

                # 要素を結合してテキスト化
                text_parts = []
                for element in res.elements:
                    text_parts.append(element.get("text", ""))

                full_text = "\n\n".join(text_parts)

                # チャンクに分割
                chunks = self._split_text(full_text, chunk_size=1000)

                # 各チャンクをベクトル化
                for i, chunk in enumerate(chunks):
                    embedding = self.embedder.encode(chunk).tolist()

                    all_documents.append(chunk)
                    all_embeddings.append(embedding)
                    all_metadatas.append({
                        "source": filename,
                        "chunk": i,
                        "file_type": os.path.splitext(filename)[1]
                    })
                    all_ids.append(f"{filename}_{i}")

                print(f"    ✓ {len(chunks)}チャンク生成")

            except Exception as e:
                print(f"    ⚠️ エラー: {e}")

        # ChromaDBに一括追加
        if all_documents:
            self.collection.add(
                documents=all_documents,
                embeddings=all_embeddings,
                metadatas=all_metadatas,
                ids=all_ids
            )

        print(f"\n✅ 合計{len(all_documents)}チャンクをインデックス化完了!\n")
        return len(all_documents)

    def search(self, query, k=3):
        """セマンティック検索"""
        if not self.collection:
            self.build_index()

        # クエリをベクトル化
        query_embedding = self.embedder.encode(query).tolist()

        # ChromaDBで検索
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=k
        )

        # 結果を整形
        formatted_results = []
        for i in range(len(results['documents'][0])):
            formatted_results.append({
                'content': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i]
            })

        return formatted_results

    def _split_text(self, text, chunk_size=1000, overlap=200):
        """テキスト分割"""
        chunks = []
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]

            if end < len(text):
                last_period = chunk.rfind('。')
                last_newline = chunk.rfind('\n')
                cut_point = max(last_period, last_newline)

                if cut_point > 0:
                    chunk = chunk[:cut_point + 1]
                    end = start + cut_point + 1

            chunks.append(chunk.strip())
            start = end - overlap

        return [c for c in chunks if c]  # 空文字を除外


# 環境変数設定
os.environ['UNSTRUCTURED_API_KEY'] = 'your-unstructured-api-key'
os.environ['OPENAI_API_KEY'] = 'your-openai-api-key'

# 使用例
kb = UnstructuredAPIKnowledgeBase()
kb.build_index()

# CrewAIと組み合わせ
search_agent = Agent(
    role='高度な検索スペシャリスト',
    goal='Unstructured APIで処理された多様なドキュメントから最適な情報を見つける',
    backstory='PDF、Word、画像など、あらゆる形式のドキュメントを扱えます',
    verbose=True,
    allow_delegation=False
)

answer_agent = Agent(
    role='回答作成スペシャリスト',
    goal='検索結果を元に、わかりやすい回答を作成',
    backstory='複雑な情報もシンプルに説明できます',
    verbose=True,
    allow_delegation=False
)

def ask_question(question):
    print(f"\n💭 質問: {question}\n")
    print("="*60)

    # 検索実行
    results = kb.search(question, k=3)

    # コンテキスト作成
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(
            f"【ソース{i}: {r['metadata']['source']}】\n{r['content']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # タスク作成
    analyze_task = Task(
        description=f"""
        以下の検索結果を分析してください:

        {context}

        質問: {question}

        重要なポイントを抽出してください。
        """,
        agent=search_agent,
        expected_output="分析結果と重要ポイント"
    )

    answer_task = Task(
        description=f"""
        質問: {question}

        分析結果を元に回答を作成してください:

        【回答】
        簡潔な答え

        【詳細説明】
        詳しい解説

        【参考情報】
        追加で役立つ情報
        """,
        agent=answer_agent,
        expected_output="構造化された回答",
        context=[analyze_task]
    )

    crew = Crew(
        agents=[search_agent, answer_agent],
        tasks=[analyze_task, answer_task],
        process=Process.sequential,
        verbose=True
    )

    result = crew.kickoff()
    return result

# 実行
answer = ask_question("Pythonのループについて教えて")
print("\n" + "="*60)
print("✅ 回答:")
print("="*60)
print(answer)

ModuleNotFoundError: No module named 'unstructured_client'

In [ ]:
!pip install -q unstructured-client sentence-transformers chromadb crewai crewai-tools openai python-dotenv
print("✅ インストール完了!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from google.colab import userdata

# Colab Secretsから安全に取得
try:
    os.environ['UNSTRUCTURED_API_KEY'] = userdata.get('UNSTRUCTURED_API_KEY')
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("✅ APIキーをColab Secretsから取得しました!")
    print(f"   🔹 Unstructured API: {os.environ['UNSTRUCTURED_API_KEY'][:8]}...")
    print(f"   🔹 OpenAI API: {os.environ['OPENAI_API_KEY'][:8]}...")
except Exception as e:
    print(f"⚠️ エラー: {e}")
    print("\n📝 設定方法:")
    print("1. 左サイドバーの🔑アイコンをクリック")
    print("2. 「新しいシークレット」をクリック")
    print("3. UNSTRUCTURED_API_KEY を追加")
    print("4. OPENAI_API_KEY を追加")
    print("5. 各シークレットで「ノートブックからのアクセスを有効にする」にチェック")

Exception in thread Thread-4 (get_input):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/crewai/events/listeners/tracing/utils.py", line 470, in get_input
    response = input().strip().lower()
               ^^^^^^^^^^^^^
AttributeError: 'dict' object has no attribute 'strip'


⚠️ エラー: Requesting secret UNSTRUCTURED_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

📝 設定方法:
1. 左サイドバーの🔑アイコンをクリック
2. 「新しいシークレット」をクリック
3. UNSTRUCTURED_API_KEY を追加
4. OPENAI_API_KEY を追加
5. 各シークレットで「ノートブックからのアクセスを有効にする」にチェック


In [16]:
import os
from google.colab import userdata

# Colab Secretsから安全に取得
try:
    os.environ['UNSTRUCTURED_API_KEY'] = userdata.get('UNSTRUCTURED_API_KEY')
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("✅ APIキーをColab Secretsから取得しました!")
    print(f"   🔹 Unstructured API: {os.environ['UNSTRUCTURED_API_KEY'][:8]}...")
    print(f"   🔹 OpenAI API: {os.environ['OPENAI_API_KEY'][:8]}...")
except Exception as e:
    print(f"⚠️ エラー: {e}")
    print("\n📝 設定方法:")
    print("1. 左サイドバーの🔑アイコンをクリック")
    print("2. 「新しいシークレット」をクリック")
    print("3. UNSTRUCTURED_API_KEY を追加")
    print("4. OPENAI_API_KEY を追加")
    print("5. 各シークレットで「ノートブックからのアクセスを有効にする」にチェック")

✅ APIキーをColab Secretsから取得しました!
   🔹 Unstructured API: Q8FjAE9X...
   🔹 OpenAI API: sk-proj-...


In [13]:
!mkdir -p documents

# サンプルファイル1
with open('documents/python_basics.txt', 'w', encoding='utf-8') as f:
    f.write("""
Pythonの基礎知識

1. 変数の宣言
Pythonでは型を明示せずに変数を宣言できます。
例: name = "太郎"、age = 25

2. リスト
複数の値を格納できるデータ構造です。
例: fruits = ["りんご", "バナナ", "オレンジ"]

3. ループ
for文とwhile文があります。

for文の例:
for i in range(5):
    print(i)  # 0, 1, 2, 3, 4 と出力

while文の例:
count = 0
while count < 5:
    print(count)
    count += 1
""")

# サンプルファイル2
with open('documents/crewai_memo.txt', 'w', encoding='utf-8') as f:
    f.write("""
CrewAIについてのメモ

CrewAIは複数のAIエージェントを協調させて
複雑なタスクを実行できるフレームワークです。

主要な概念:
- Agent: 特定の役割を持つAI (例: 検索担当、分析担当)
- Task: エージェントが実行する具体的な作業
- Crew: エージェントのチームを管理するマネージャー
- Tools: エージェントが使用できる機能(検索、計算など)

利点:
- 複雑なタスクを分割して処理できる
- エージェント間で協力・連携が可能
- 柔軟な設定とカスタマイズ
""")

print("✅ サンプルドキュメント作成完了!")
print("\n📁 作成されたファイル:")
!ls -lh documents/

✅ サンプルドキュメント作成完了!

📁 作成されたファイル:
total 8.0K
-rw-r--r-- 1 root root 600 Jan 10 06:27 crewai_memo.txt
-rw-r--r-- 1 root root 465 Jan 10 06:27 python_basics.txt


In [17]:
import os
import chromadb
from unstructured_client import UnstructuredClient
from unstructured_client.models import shared, operations # Import operations
from langchain_openai import OpenAIEmbeddings # Reverted to OpenAIEmbeddings for stability
from crewai import Agent, Task, Crew, Process

class UnstructuredAPIKnowledgeBase:
    """Unstructured APIを使った知識ベース"""

    def __init__(self, docs_path='./documents'):
        self.docs_path = docs_path

        # Unstructured APIクライアント
        print("🔗 Unstructured APIに接続中...")
        self.unstructured_client = UnstructuredClient(
            api_key_auth=os.getenv("UNSTRUCTURED_API_KEY")
        )

        # EmbeddingモデルをOpenAIEmbeddingsに切り替え
        print("🤖 Embeddingモデルをロード中... (OpenAI)")
        # Explicitly pass the API key
        self.embedder = OpenAIEmbeddings(openai_api_key=os.getenv("OPENAI_API_KEY"))

        # ChromaDB
        self.chroma_client = chromadb.PersistentClient(path="./chroma_db")
        self.collection = None
        print("✅ 初期化完了!\n")

    def build_index(self):
        """インデックス構築"""
        print("📚 ドキュメント処理開始...\n")

        # コレクション準備
        try:
            self.chroma_client.delete_collection("documents")
        except:
            pass

        self.collection = self.chroma_client.create_collection(
            name="documents"
        )

        all_documents = []
        all_embeddings = []
        all_metadatas = []
        all_ids = []

        # 各ファイルを処理
        for filename in os.listdir(self.docs_path):
            filepath = os.path.join(self.docs_path, filename)

            try:
                print(f"  🔄 処理中: {filename}")

                # ファイルを読み込んで送信
                with open(filepath, "rb") as f:
                    file_content = f.read()

                # unstructured_client.general.partitionの呼び出しを修正
                # requestオブジェクトを作成し、それを渡す
                request_obj = operations.PartitionRequest(
                    partition_parameters=shared.PartitionParameters(
                        files=shared.Files(
                            content=file_content,
                            file_name=filename,
                        ),
                        strategy="fast",  # Colabでは高速モード推奨
                        languages=["jpn", "eng"],
                    )
                )
                res = self.unstructured_client.general.partition(request=request_obj)

                # テキスト抽出
                text_parts = [
                    element.get("text", "")
                    for element in res.elements
                    if element.get("text")
                ]
                full_text = "\n\n".join(text_parts)

                # チャンク分割
                chunks = self._split_text(full_text)

                # ベクトル化
                for i, chunk in enumerate(chunks):
                    # OpenAIEmbeddingsを使用
                    embedding = self.embedder.embed_query(chunk)

                    all_documents.append(chunk)
                    all_embeddings.append(embedding)
                    all_metadatas.append({
                        "source": filename,
                        "chunk": i
                    })
                    all_ids.append(f"{filename}_{i}")

                print(f"    ✓ {len(chunks)}チャンク生成")

            except Exception as e:
                print(f"    ⚠️ エラー: {str(e)[:100]}")

        # ChromaDBに追加
        if all_documents:
            self.collection.add(
                documents=all_documents,
                embeddings=all_embeddings,
                metadatas=all_metadatas,
                ids=all_ids
            )
            print(f"\n✅ 合計{len(all_documents)}チャンクをインデックス化完了!\n")
        else:
            print("\n⚠️ ドキュメントが見つかりませんでした")

        return len(all_documents)

    def search(self, query, k=3):
        """検索"""
        if not self.collection:
            self.build_index()

        # クエリをベクトル化
        query_embedding = self.embedder.embed_query(query)

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=k
        )

        formatted = []
        for i in range(len(results['documents'][0])):
            formatted.append({
                'content': results['documents'][0][i],
                'metadata': results['metadatas'][0][i]
            })

        return formatted

    def _split_text(self, text, chunk_size=1000, overlap=200):
        """テキスト分割"""
        chunks = []
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]

            if end < len(text):
                last_period = chunk.rfind('。')
                last_newline = chunk.rfind('\n')
                cut_point = max(last_period, last_newline)

                if cut_point > 0:
                    end = start + cut_point + 1
                    chunk = text[start:end]

            if chunk.strip():
                chunks.append(chunk.strip())
            start = end - overlap

        return chunks

# 初期化
kb = UnstructuredAPIKnowledgeBase()
print("システム準備完了!")

🔗 Unstructured APIに接続中...
🤖 Embeddingモデルをロード中... (OpenAI)
✅ 初期化完了!

システム準備完了!


In [ ]:
import transformers, sys
print("python:", sys.version)
print("transformers:", transformers.__version__)
print("transformers path:", transformers.__file__)


python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
transformers: 4.46.3
transformers path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [15]:
!pip uninstall -y langchain langchain-core transformers sentence-transformers openai crewai crewai-tools langchain-openai chromadb unstructured-client pypdf python-docx python-dotenv
!pip install -q unstructured-client sentence-transformers chromadb crewai crewai-tools openai python-dotenv langchain-openai
print("✅ インストール完了!")

Found existing installation: langchain-core 0.3.82
Uninstalling langchain-core-0.3.82:
  Successfully uninstalled langchain-core-0.3.82
Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: sentence-transformers 5.2.0
Uninstalling sentence-transformers-5.2.0:
  Successfully uninstalled sentence-transformers-5.2.0
Found existing installation: openai 1.83.0
Uninstalling openai-1.83.0:
  Successfully uninstalled openai-1.83.0
Found existing installation: crewai 1.8.0
Uninstalling crewai-1.8.0:
  Successfully uninstalled crewai-1.8.0
Found existing installation: crewai-tools 1.8.0
Uninstalling crewai-tools-1.8.0:
  Successfully uninstalled crewai-tools-1.8.0
Found existing installation: langchain-openai 0.3.23
Uninstalling langchain-openai-0.3.23:
  Successfully uninstalled langchain-openai-0.3.23
Found existing installation: chromadb 1.1.1
Uninstalling chromadb-1.1.1:
  Successfully un

In [ ]:
import os
import chromadb
from unstructured_client import UnstructuredClient
from unstructured_client.models import shared
from sentence_transformers import SentenceTransformer
from crewai import Agent, Task, Crew, Process

class UnstructuredAPIKnowledgeBase:
    """Unstructured APIを使った知識ベース"""

    def __init__(self, docs_path='./documents'):
        self.docs_path = docs_path

        # Unstructured APIクライアント
        print("🔗 Unstructured APIに接続中...")
        self.unstructured_client = UnstructuredClient(
            api_key_auth=os.getenv("UNSTRUCTURED_API_KEY")
        )

        # ローカルEmbeddingモデル
        print("🤖 Embeddingモデルをロード中...")
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

        # ChromaDB
        self.chroma_client = chromadb.PersistentClient(path="./chroma_db")
        self.collection = None
        print("✅ 初期化完了!\n")

    def build_index(self):
        """インデックス構築"""
        print("📚 ドキュメント処理開始...\n")

        # コレクション準備
        try:
            self.chroma_client.delete_collection("documents")
        except:
            pass

        self.collection = self.chroma_client.create_collection(
            name="documents"
        )

        all_documents = []
        all_embeddings = []
        all_metadatas = []
        all_ids = []

        # 各ファイルを処理
        for filename in os.listdir(self.docs_path):
            filepath = os.path.join(self.docs_path, filename)

            try:
                print(f"  🔄 処理中: {filename}")

                # ファイルを読み込んで送信
                with open(filepath, "rb") as f:
                    file_content = f.read()

                req = shared.PartitionParameters(
                    files=shared.Files(
                        content=file_content,
                        file_name=filename,
                    ),
                    strategy="fast",  # Colabでは高速モード推奨
                    languages=["jpn", "eng"],
                )

                # API実行
                res = self.unstructured_client.general.partition(req)

                # テキスト抽出
                text_parts = [
                    element.get("text", "")
                    for element in res.elements
                    if element.get("text")
                ]
                full_text = "\n\n".join(text_parts)

                # チャンク分割
                chunks = self._split_text(full_text)

                # ベクトル化
                for i, chunk in enumerate(chunks):
                    embedding = self.embedder.encode(chunk).tolist()

                    all_documents.append(chunk)
                    all_embeddings.append(embedding)
                    all_metadatas.append({
                        "source": filename,
                        "chunk": i
                    })
                    all_ids.append(f"{filename}_{i}")

                print(f"    ✓ {len(chunks)}チャンク生成")

            except Exception as e:
                print(f"    ⚠️ エラー: {str(e)[:100]}")

        # ChromaDBに追加
        if all_documents:
            self.collection.add(
                documents=all_documents,
                embeddings=all_embeddings,
                metadatas=all_metadatas,
                ids=all_ids
            )
            print(f"\n✅ 合計{len(all_documents)}チャンクをインデックス化完了!\n")
        else:
            print("\n⚠️ ドキュメントが見つかりませんでした")

        return len(all_documents)

    def search(self, query, k=3):
        """検索"""
        if not self.collection:
            self.build_index()

        query_embedding = self.embedder.encode(query).tolist()

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=k
        )

        formatted = []
        for i in range(len(results['documents'][0])):
            formatted.append({
                'content': results['documents'][0][i],
                'metadata': results['metadatas'][0][i]
            })

        return formatted

    def _split_text(self, text, chunk_size=1000, overlap=200):
        """テキスト分割"""
        chunks = []
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]

            if end < len(text):
                last_period = chunk.rfind('。')
                last_newline = chunk.rfind('\n')
                cut_point = max(last_period, last_newline)

                if cut_point > 0:
                    end = start + cut_point + 1
                    chunk = text[start:end]

            if chunk.strip():
                chunks.append(chunk.strip())
            start = end - overlap

        return chunks

# 初期化
kb = UnstructuredAPIKnowledgeBase()
print("システム準備完了!")

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
No module named 'transformers.modeling_layers'

In [ ]:
python# ドキュメントをインデックス化
num_chunks = kb.build_index()
print(f"\n🎉 {num_chunks}個のチャンクが検索可能になりました!")

NameError: name 'python' is not defined

In [14]:
# ドキュメントをインデックス化
num_chunks = kb.build_index()
print(f"\n🎉 {num_chunks}個のチャンクが検索可能になりました!")

📚 ドキュメント処理開始...

  🔄 処理中: crewai_memo.txt
    ✓ 1チャンク生成
  🔄 処理中: python_basics.txt
    ✓ 1チャンク生成

✅ 合計2チャンクをインデックス化完了!


🎉 2個のチャンクが検索可能になりました!


In [18]:
import os
from langchain_openai import ChatOpenAI # Import ChatOpenAI for the agents' LLM
from crewai import Agent

# Explicitly define the LLM for agents
agent_llm = ChatOpenAI(model="gpt-4", openai_api_key=os.getenv("OPENAI_API_KEY")) # Using gpt-4 as a powerful model, can be changed.

# Agent 1: 検索スペシャリスト
search_agent = Agent(
    role='検索スペシャリスト',
    goal='質問に関連する情報を見つける',
    backstory='Unstructured APIで処理された多様なドキュメントから最適な情報を抽出できます',
    verbose=True,
    allow_delegation=False,
    llm=agent_llm # Explicitly assign the LLM
)

# Agent 2: 回答作成スペシャリスト
answer_agent = Agent(
    role='回答作成スペシャリスト',
    goal='わかりやすい回答を作成する',
    backstory='複雑な情報もシンプルに説明できます',
    verbose=True,
    allow_delegation=False,
    llm=agent_llm # Explicitly assign the LLM
)

print("✅ エージェント準備完了!")

✅ エージェント準備完了!


In [19]:
from crewai import Task, Crew, Process

def ask_question(question):
    """質問に回答"""
    print(f"\n💭 質問: {question}")
    print("="*60 + "\n")

    # 検索
    results = kb.search(question, k=3)

    # コンテキスト作成
    context_parts = [
        f"【{r['metadata']['source']}】\n{r['content']}"
        for r in results
    ]
    context = "\n\n---\n\n".join(context_parts)

    # タスク作成
    analyze_task = Task(
        description=f"""
        以下の情報を分析:
        {context}

        質問: {question}
        重要なポイントを抽出してください。
        """,
        agent=search_agent,
        expected_output="分析結果"
    )

    answer_task = Task(
        description=f"""
        質問: {question}

        回答を作成してください:
        - 簡潔な答え
        - 詳しい説明
        - 補足情報
        """,
        agent=answer_agent,
        expected_output="回答",
        context=[analyze_task]
    )

    # 実行
    crew = Crew(
        agents=[search_agent, answer_agent],
        tasks=[analyze_task, answer_task],
        process=Process.sequential,
        verbose=True
    )

    return crew.kickoff()

print("✅ 質問応答システム準備完了!")

✅ 質問応答システム準備完了!


In [ ]:
# 質問してみる
answer = ask_question("Pythonのfor文について教えて")

print("\n" + "="*60)
print("✅ 回答:")
print("="*60)
print(answer)


💭 質問: Pythonのfor文について教えて



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  5a9a09a2-7c25-4551-aebb-ed5a605fe350                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          以下の情報を分析:                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│          質問: Pythonのfor文について教えて                                                                      │
│          重要なポイントを抽出してください。                                                                     │
│                                                                                                                 │
│  ID: ee761680-7332-473c-a6c9-e56e82d88445                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 検索スペシャリスト                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│          以下の情報を分析:                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│          質問: Pythonのfor文について教えて                                                                      │
│          重要なポイントを抽出してください。                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 検索スペシャリスト                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Pythonのfor文についての重要なポイントは以下の通りです。                                                        │
│                                                                                                                 │
│  1. 基本構文                                                                                                    │
│  Pythonのfor文はイテラブル（リスト・タプル・文字列・辞書・rangeオブジェクトなど）から要素を一つずつ取り出して   │
│  処理を繰り返します。                                                                                           │
│  ```                                                                                                            │
│  for 変数 in イテラブル:                                                                                        │
│      処理                                                                                                       │
│  ```                                                                                                            │
│  例:                                                                                                            │
│  ```python                                                                                                      │
│  for i in [1, 2, 3]:                                                                                            │
│      print(i)                                                                                                   │
│  ```                                                                                                            │
│                                                                                                                 │
│  2. イテラブルとは                                                                                              │
│  イテラブルとは反復可能なオブジェクトのことで、リスト・タプル・文字列・辞書のキー・値などがこれにあたります。   │
│                                                                                                                 │
│  3. range関数の利用                                                                                             │
│  整数の連続した範囲を生成するためにrange()関数がよく使われます。                                                │
│  ```                                                                                                            │
│  for i in range(5):                                                                                             │
│      print(i)                                                                                                   │
│  ```                                                                                                            │
│  0から4までの5回繰り返しを行います。                                                                            │
│                                                                                                                 │
│  4. enumerate関数の利用                                                                                         │
│  繰り返しの中で要素のインデックスも取得したい場合、enumerate()が便利です。                                      │
│  ```python                                                                                                      │
│  for index, value in enumerate(['a', 'b', 'c']):                                                                │
│      print(index, value)                                                                                        │
│  ```                                                                    

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 回答作成スペシャリスト                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Pythonのfor文について、簡潔な答えと詳しい説明、そして補足情報を以下にまとめます。                              │
│                                                                                                                 │
│  【簡潔な答え】                                                                                                 │
│  Pythonのfor文は「イテラブル（反復可能なオブジェクト）から要素を一つずつ取り出して繰り返し処理する」ための構文  │
│  です。                                                                                                         │
│                                                                                                                 │
│  【詳しい説明】                                                                                                 │
│  1. 基本構文                                                                                                    │
│  ```python                                                                                                      │
│  for 変数 in イテラブル:                                                                                        │
│      処理                                                                                                       │
│  ```                                                                                                            │
│  ここで「イテラブル」とは、リスト・タプル・文字列・辞書・rangeなど、順番に要素を取り出せるオブジェクトのことで  │
│  す。                                                                                                           │
│  例えば、リスト[1, 2, 3]を使う場合、                                                                            │
│  ```python                                                                                                      │
│  for i in [1, 2, 3]:                                                                                            │
│      print(i)                                                                                                   │
│  ```                                                                                                            │
│  と書くと、1, 2, 3が順番に表示されます。                                                                        │
│                                                                                                                 │
│  2. range関数                                                                                                   │
│  range(n)は0からn-1までの整数を生成し、                                                                         │
│  ```python                                                                                                      │
│  for i in range(5):                                                                                             │
│      print(i)                                                                                                   │
│  ```                                                                                                            │
│  は0〜4までの数字を出力します。数値の繰り返しに便利です。                                                       │
│                                                                                                                 │
│  3. enumerate関数                                                                                               │
│  要素に加えてインデックス（何番目か）も取得したい時に使います。                                                 │
│  ```python                                                                                                  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│          質問: Pythonのfor文について教えて                                                                      │
│                                                                                                                 │
│          回答を作成してください:                                                                                │
│          - 簡潔な答え                                                                                           │
│          - 詳しい説明                                                                                           │
│          - 補足情報                                                                                             │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  回答作成スペシャリスト                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ 回答:
Pythonのfor文について、簡潔な答えと詳しい説明、そして補足情報を以下にまとめます。

【簡潔な答え】
Pythonのfor文は「イテラブル（反復可能なオブジェクト）から要素を一つずつ取り出して繰り返し処理する」ための構文です。

【詳しい説明】
1. 基本構文  
```python
for 変数 in イテラブル:
    処理
```
ここで「イテラブル」とは、リスト・タプル・文字列・辞書・rangeなど、順番に要素を取り出せるオブジェクトのことです。  
例えば、リスト[1, 2, 3]を使う場合、
```python
for i in [1, 2, 3]:
    print(i)
```
と書くと、1, 2, 3が順番に表示されます。

2. range関数  
range(n)は0からn-1までの整数を生成し、
```python
for i in range(5):
    print(i)
```
は0〜4までの数字を出力します。数値の繰り返しに便利です。

3. enumerate関数  
要素に加えてインデックス（何番目か）も取得したい時に使います。
```python
for index, value in enumerate(['a', 'b', 'c']):
    print(index, value)
```

4. 辞書のfor文  
辞書のキーだけをループする場合、
```python
for key in {'a':1, 'b':2}:
    print(key)
```
キーと値の両方を使いたい場合、
```python
for key, value in {'a':1, 'b':2}.items():
    print(key, value)
```

5. ネストしたfor文（多重ループ）  
for文の中にさらにfor文を書いて複雑な繰り返しも可能です。
```python
for i in range(2):
    for j in range(3):
        print(i, j)
```

6. breakとcontinue  
- breakはループを途中で終了します。  
- continueはその回の処理をスキップして次へ進みます。  
```python
for i in r

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

In [22]:
# from google.colab import drive

# # Google Driveをマウント
# drive.mount('/content/drive')

# # Google Drive内のフォルダを指定
# drive_docs = '/content/drive/MyDrive/非構造化データ' # ユーザーの指定に合わせてパスを修正

# # そのフォルダをナレッジベースに
# kb_drive = UnstructuredAPIKnowledgeBase(docs_path=drive_docs)
# kb_drive.build_index()

# print("✓ Google Driveのドキュメントが使えるようになりました!")
print("Google Driveとの連携は解除されました。")

Google Driveとの連携は解除されました。


In [23]:
# if 'kb_drive' in locals() and kb_drive.collection:
#     print(f"✅ Google Driveドキュメントのインデックス化が完了しました！")
#     print(f"   合計 {kb_drive.collection.count()} 個のチャンクが検索可能です。")
# else:
#     print("⚠️ Google Driveドキュメントのインデックス化はまだ完了していないか、エラーが発生しています。")
#     print("   前回の実行出力を確認してください。")
print("Google Drive関連のチェックはスキップされました。")

Google Drive関連のチェックはスキップされました。


In [ ]:
answer = ask_question("AIで重要なことは")

print("\n" + "="*60)
print("✅ 回答:")
print("="*60)
print(answer)

NameError: name 'ask_question' is not defined

In [20]:
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import Layout

print("✅ UI関連のライブラリをインポートしました")

✅ UI関連のライブラリをインポートしました


In [21]:
# UI要素の作成
question_input = widgets.Textarea(
    value='',
    placeholder='ここに質問を入力してください...', # 質問の例を追加
    description='質問:',
    disabled=False,
    layout=Layout(width='auto', height='80px')
)

submit_button = widgets.Button(
    description='回答を生成',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='質問に対する回答をAIに生成させます',
    icon='comment-dots'
)

output_area = widgets.Output()

# ボタンがクリックされたときの処理
def on_button_click(b):
    with output_area:
        output_area.clear_output()
        question = question_input.value
        if question.strip():
            print(f"🤔 質問: {question}\n")
            print("🔍 AI Agentが処理中...\n")
            print("="*50)

            try:
                answer = ask_question(question)
                print("="*50)
                print("\n✅ 回答:")
                print(answer)
            except Exception as e:
                print(f"⚠️ エラーが発生しました: {e}")
                print("APIキーが正しく設定されているか、またはAPIのクォータを超過していないか確認してください。")
        else:
            print("質問を入力してください。")

submit_button.on_click(on_button_click)

# UIの表示
display(HTML("<h2>質問応答システム</h2>"))
display(question_input, submit_button, output_area)

print("✅ 質問応答UIが作成されました！")

Textarea(value='', description='質問:', layout=Layout(height='80px', width='auto'), placeholder='ここに質問を入力してください.…

Button(button_style='info', description='回答を生成', icon='comment-dots', style=ButtonStyle(), tooltip='質問に対する回答をA…

Output()

✅ 質問応答UIが作成されました！


### 使い方
上のテキストボックスに質問を入力し、「回答を生成」ボタンをクリックしてください。AIがドキュメントを検索し、質問に対する回答を生成します。

In [ ]:
# 質問してみる
answer = ask_question("AIで重要なことを教えて")

print("\n" + "="*60)
print("✅ 回答:")
print("="*60)
print(answer)

NameError: name 'ask_question' is not defined

In [1]:
!pip -q install fastapi uvicorn nest-asyncio

from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import uvicorn

app = FastAPI()

class RunReq(BaseModel):
    input: str

@app.post("/run")
def run(req: RunReq):
    # ここにあなたのAI Agent実行コードを呼ぶ
    result = ask_question(req.input) # ask_question関数を呼び出す
    return {"output": result}

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)

SyntaxError: invalid syntax (ipython-input-2691780803.py, line 2)

In [24]:
!pip -q install fastapi uvicorn nest-asyncio

from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import uvicorn

app = FastAPI()

class RunReq(BaseModel):
    input: str

@app.post("/run")
def run(req: RunReq):
    # ここにあなたのAI Agent実行コードを呼ぶ
    # result = agent.invoke(req.input)
    result = f"agent result: {req.input}"
    return {"output": result}

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)


RuntimeError: asyncio.run() cannot be called from a running event loop

In [25]:
%%writefile server.py
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class RunReq(BaseModel):
    input: str

@app.post("/run")
def run(req: RunReq):
    # ここであなたのAI Agentを呼ぶ
    # result = agent.invoke(req.input)
    return {"output": f"agent result: {req.input}"}


Writing server.py


In [26]:
!pip -q install fastapi uvicorn
!uvicorn server:app --host 0.0.0.0 --port 8000


INFO:     Started server process [8932]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [8932]


In [27]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared tunnel --url http://localhost:8000


Selecting previously unselected package cloudflared.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2025.11.1) ...
Setting up cloudflared (2025.11.1) ...
Processing triggers for man-db (2.10.2-1) ...
2026-01-10T06:58:04Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-01-10T06:58:04Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-01-10T06:58:08Z